## Setup

In [1]:
from google.colab import userdata
access_token = userdata.get('CASM-NER')

In [2]:
%%capture
!pip install transformers
!pip install sentencepiece
!pip install seqeval
!pip install datasets
# !pip install git+https://github.com/ay94/multilingual-ner.git

In [ ]:
# from ner import evaluation

In [3]:
## Mount GDrive
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

## Imports
import os
import sys
import nltk
import time
import torch
import random
import subprocess
import numpy as np
import pandas as pd
import datetime as dt
from itertools import groupby
from tqdm.notebook import tqdm
from datasets import load_dataset
from transformers import pipeline
from collections import Counter, defaultdict
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForTokenClassification, AutoTokenizer
from seqeval.metrics import f1_score as seq_f1, precision_score as seq_precision, recall_score as seq_recall, classification_report as seq_classification
from sklearn.metrics import f1_score as skl_f1, precision_score as skl_precision, recall_score as skl_recall, classification_report as skl_classification

Mounted at /content/drive/


In [4]:
# Append the library files into the notebook system path for import
sys.path.append('/content/drive/Shareddrives/Machine Translation/Model benchmarking/Libraries/1.0.2')
# import custom library files
import ner, utils

## Load datasets

### Wikiann

In [5]:
wikiann_label_map = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6
}

wikiann = ner.ReadNERData()
wikiann_words, wikiann_labels = wikiann.read_dataset(
    'wikiann',
    wikiann_label_map,
    lang='ja'
)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/datasets/load.py:1429: FutureWarning: The repository for wikiann contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/wikiann
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating test Split


  0%|          | 0/10000 [00:00<?, ?it/s]

# Evaluate model

In [7]:
# alignment = {
#     'O': 'O',
#     'PER': 'PER',
#     'ORG': 'ORG',
#     'ORG-P': 'ORG',
#     'ORG-O': 'ORG',
#     'LOC': 'LOC',
#     'INS': 'O',
#     'PRD': 'O',
#     'EVT': 'O',
# }

model_name = "ken11/bert-japanese-ner"
model_name_output = 'ken11/bert-japanese-ner'
model_evaluation = ner.ModelEvaluation(
    model_name,
    # alignment
)

OSError: Can't load tokenizer for 'ken11/bert-japanese-ner'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'ken11/bert-japanese-ner' is the correct path to a directory containing all relevant files for a BertTokenizerFast tokenizer.

In [ ]:
print(model_evaluation.model.config.id2label)

{0: 'O', 1: 'PER', 2: 'ORG', 3: 'ORG-P', 4: 'ORG-O', 5: 'LOC', 6: 'INS', 7: 'PRD', 8: 'EVT'}


In [ ]:
data_name = "wikiann"
wikiann_evaluation_output = model_evaluation.evaluate_model(wikiann_words, wikiann_labels)

  0%|          | 0/625 [00:00<?, ?it/s]

In [ ]:
wikiann_seqeval = wikiann_evaluation_output.get_classification('Seqeval')
wikiann_seqeval